# Chapter 17 — Markov chains

Companion notebook to `docs/models/ch17_markov.md` and the Streamlit page `app/pages/17_Markov_Chains.py`.

The Markov Convergence Theorem: an irreducible aperiodic chain converges to a unique stationary distribution regardless of where it started. History doesn't matter in the long run — only the transition probabilities do.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt
import numpy as np

from modelthinker.dynamics.markov import (
    PRESETS,
    distribution_over_time,
    intervention_experiment,
    stationary_distribution,
)

## 1. Compute the stationary distribution for each preset

In [ ]:
for spec in PRESETS.values():
    pi = stationary_distribution(spec['P'])
    shares = ', '.join(f'{s}={100*p:.1f}%' for s, p in zip(spec['states'], pi, strict=True))
    print(f"{spec['name']}\n  π: {shares}\n")

## 2. Different starts, same end (Markov Convergence Theorem)

In [ ]:
spec = PRESETS['weather']
P = spec['P']
pi = stationary_distribution(P)

starts = [
    ('100% sunny start',  np.array([1.0, 0.0, 0.0])),
    ('100% rainy start',  np.array([0.0, 0.0, 1.0])),
    ('Uniform start',      np.array([1/3, 1/3, 1/3])),
]

fig, ax = plt.subplots(figsize=(9, 4))
for label, x0 in starts:
    dist = distribution_over_time(P, initial_distribution=x0, steps=80)
    ax.plot(dist[:, 0], label=label, lw=2)   # % sunny over time
ax.axhline(pi[0], color='red', ls='--', label=f'π (sunny) = {pi[0]:.3f}')
ax.set(xlabel='step', ylabel='% sunny', title='Three starts, one destination', ylim=(0, 1))
ax.legend(); plt.show()

## 3. Intervention vs rule change

The headline result: dumping everyone into a state is temporary; changing the transition matrix is permanent.

In [ ]:
spec = PRESETS['drug']
P = spec['P']

# Rule change: reduce sober→using transitions by 8%, move that weight back to sober self-loop
P_after = P.copy()
moved = 0.08
P_after[0, 1] -= moved
P_after[0, 0] += moved

res = intervention_experiment(
    P=P, steps=400, intervention_step=100,
    intervention_state=0,   # move everyone into 'Sober'
    P_after=P_after,
)

focus = 2   # 'Addicted' state
labels = spec['states']
pi_before = stationary_distribution(P)
pi_after = stationary_distribution(P_after)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(res.time_axis, res.baseline[:, focus], color='gray', lw=1.5, ls=':', label='baseline (no intervention)')
ax.plot(res.time_axis, res.intervention[:, focus], color='#F76707', lw=2, label='one-off intervention at t=100')
ax.plot(res.time_axis, res.rule_change[:, focus], color='#37B24D', lw=2, label='rule change at t=100')
ax.axhline(pi_before[focus], color='red', ls='--', label=f'original π = {pi_before[focus]:.3f}')
ax.axhline(pi_after[focus], color='green', ls='--', label=f'new π = {pi_after[focus]:.3f}')
ax.axvline(100, color='black', ls=':', alpha=0.4)
ax.set(xlabel='step', ylabel=f'fraction {labels[focus]}',
       title='One-off interventions decay. Rule changes stick.')
ax.legend(loc='upper right', fontsize=9); plt.show()

## 4. What-ifs to try

1. Compute the stationary distribution for the PageRank preset. Which page has the highest 'authority score'?
2. Take the students preset and construct a **P_after** where 'bored' is a self-loop of 0.99. What happens to π? What does that tell you about the pedagogical intervention?
3. Simulate the drug chain from 100% 'Sober' vs 100% 'Addicted' — how many steps does it take for the two population distributions to agree within 1 percentage point?